# CartPole: Q-table, DQN (Keras) and A2C, with every line tied to its explanation

**This notebook puts the connections *inside the code comments*,** so you can read a cell on its own without scrolling back and forth. Every code cell uses these tags:

| Tag in a comment | Meaning |
|---|---|
| `# WHAT:` | the line below, in plain words |
| `# WHY:` | why that line exists, its significance |
| `# SAME AS:` | the same idea written differently somewhere else. **I quote the other line right there**, so you don't have to go find it |
| `# BECOMES:` | what this line turns into in the next, fancier version |
| `# WATCH:` | a trap or a detail people trip over |

Each part starts with a **"You are here"** box and each code cell has a short note before it saying what to look for in the output. Output numbers are printed by the code itself, so the sentences next to them always match what you ran. Everything uses random numbers, so your numbers will differ a little from run to run. The whole notebook takes roughly 10 minutes to run (the two Keras trainings are the slow parts).

### The five jobs every one of these programs does

Every method in this notebook (a table, a Keras DQN, A2C) does the same five jobs. Only *how* each job is done changes. Keep this table in mind; every part fills in one column of it.

| # | Job | Plain meaning |
|---|---|---|
| 1 | **Environment loop** | Show the agent a state, take its action, get back a reward and the next state |
| 2 | **Policy** | The rule that turns a state into an action |
| 3 | **Target** | The "answer key": what the number *should* have been, now that we've seen what really happened |
| 4 | **Update** | Move the learned numbers toward the target |
| 5 | **Exploration** | Sometimes try something other than the current best, so we can discover better options |

### Symbols used in comments
`state` = what the agent sees now · `action` = 0 (push left) or 1 (push right) · `reward` = points from the step just taken · `next_state` = what it sees after · `gamma` = how much the future counts (0 to 1) · `alpha` = how big a step the update takes · `epsilon` = chance of picking a random action

# Part 1: What is a policy, and why is there a coin flip?

> **You are here:** nothing learned yet. This part only builds a ruler for measuring how good an agent is.
> **New idea:** a *policy* is any rule that turns a state into an action, and the score it earns is how we judge it.
> **Why a coin flip is in here:** it's the *starting point of every learner in this notebook*. An agent that has learned nothing behaves like a coin flip. So its score is the **"before"** number. Everything we build later is trying to move the score from the coin-flip number up toward a good policy's number. Without the "before" number you couldn't tell whether learning did anything.

Below there are two policies plus one game and one scoring function:

- `coin_flip_policy` is the **"before"** (a learner that knows nothing).
- `hand_rule_policy` is a rule I wrote by hand. It is the **"proof a good policy exists"**, and it tells us what score is possible. In real problems nobody can write this rule by hand, which is exactly why we need learning.
- `play_one_game(policy)` is the **ruler**. Any policy can be plugged in, and it returns the score.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # WHAT: hide TensorFlow's noisy startup messages (it only gets loaded in Part 4)
import random, collections
import numpy as np

random.seed(5)                                # WHAT: fix the random numbers so this cell prints the same thing each run


class DummyGame:
    # WHAT: a made-up game. State = ONE number, `tilt`. Start at 0. Reaching +3 or -3 means you fell.
    # WHY:  real games (CartPole) work the same way but have 4 state numbers instead of 1.

    def reset(self):
        self.tilt = 0
        self.ticks = 0
        return self.tilt                      # SAME AS: CartPole's `obs, info = env.reset()` (hands back the first state)

    def step(self, action):                   # action: 0 = push left, 1 = push right
        drift = random.choice([-1, 1])        # WHAT: the world randomly shoves the tilt
        push = -1 if action == 0 else +1      # WHAT: your push
        self.tilt = self.tilt + drift + push  # WHAT: new tilt = old tilt + world's shove + your push
        self.ticks = self.ticks + 1

        reward = 1                            # SAME AS: CartPole's +1 for every tick the pole is still up
        terminated = abs(self.tilt) >= 3      # WHAT: True if you FELL (the game ended because you failed)
        truncated = self.ticks >= 50          # WHAT: True if TIME ran out (the game ended, but not your fault)
        return self.tilt, reward, terminated, truncated
        # SAME AS: CartPole's `obs, reward, terminated, truncated, info = env.step(action)` (CartPole adds an `info` extra)


def coin_flip_policy(state):
    # WHAT: ignores the state entirely and picks left or right at random.
    # WHY:  THIS IS THE "BEFORE" SCORE. A learner that has learned nothing acts like this.
    # BECOMES: in Part 3 the table is all zeros at the start, so its choices are coin flips.
    # BECOMES: in Part 4 an untrained network gives near-random outputs, and epsilon starts at 1.0 (= always random).
    return random.choice([0, 1])


def hand_rule_policy(state):
    # WHAT: if tilt is positive, push left; otherwise push right. (Always push toward zero.)
    # WHY:  PROOF A GOOD POLICY EXISTS, and it shows the score we're aiming at.
    # WATCH: I wrote this by looking at the game and thinking. Learning means finding a rule like this AUTOMATICALLY.
    if state > 0:
        return 0
    else:
        return 1


def play_one_game(policy):
    # WHAT: play one full game using whatever `policy` function we're handed, and return the total reward.
    game = DummyGame()
    state = game.reset()
    total = 0
    while True:
        action = policy(state)   # WHY: THE PLUG-IN POINT. This one line is the only thing that differs between a coin flip,
                                 #      a hand rule, a Q-table, a Keras network and A2C. EVERY algorithm in this notebook
                                 #      is just a different way of filling in this line.
        state, reward, terminated, truncated = game.step(action)
        total = total + reward
        if terminated or truncated:
            return total


coin = [play_one_game(coin_flip_policy) for _ in range(1000)]   # WHAT: 1000 games with the "knows nothing" player
rule = [play_one_game(hand_rule_policy) for _ in range(1000)]   # WHAT: 1000 games with the "good rule" player

print(f"coin flip (the BEFORE score, knows nothing)  : {np.mean(coin):5.1f} out of 50")
print(f"hand rule (shows a good policy exists)       : {np.mean(rule):5.1f} out of 50")
print(f"=> the gap learning has to close: {np.mean(rule) - np.mean(coin):.1f} points")

**What the output means.** The coin flip scores low because it falls over quickly. The hand rule scores the maximum. The difference between those two numbers is **the whole job of learning**: start at the coin-flip level and end near the rule's level, *without* anyone telling the agent the rule. In the next part we'll watch a table do exactly that on an even smaller game, so you can check every number by hand.

# Part 2: The Q-table, by hand, on a tiny hallway

> **You are here:** you have a ruler (Part 1). Now we build the first real learner.
> **New idea:** a Q-table (a spreadsheet of scores) and the one-cell update rule.
> **Same as before:** policy = "a rule from state to action" (here the rule is "pick the biggest number in this state's row").

**The hallway.** Four squares in a line: `S0 → S1 → S2 → GOAL`. The walker starts on S0 and each tick moves left or right (the left wall at S0 means pushing left there just keeps you on S0). Stepping onto GOAL ends the game and pays **reward 10**. Every other step pays **0**. Squares S0, S1, S2 are the "states"; I'm only naming them so the printouts are readable.

**The Q-table** is a spreadsheet with one row per state and one column per action (left / right). Each cell holds one number: *"if I'm in this row's state and take this column's action, about how many points will I collect from here to the end?"* It starts at 0 everywhere because the walker knows nothing.

**The one rule (the update).** After every step, change exactly one cell, the (state, action) you just used:

1. `target = reward + gamma * (biggest number in the NEXT state's row)`. This is the **answer key**: "one real reward I just saw, plus what the next row says is still to come."
2. `new = old + alpha * (target - old)`. This **moves the cell part of the way to the answer key**.

The cell below runs three laps where the walker always pushes right, and prints every number in those two formulas so you can check the arithmetic.

In [ ]:
class Hallway:
    # WHAT: squares 0, 1, 2 and then GOAL (= 3). Reaching 3 ends the game with reward 10.
    # WHY:  this is a tiny MDP: states (0,1,2), actions (left/right), rewards (10 at the goal, else 0).
    def reset(self):
        self.pos = 0
        return self.pos

    def step(self, action):
        if action == 1:
            self.pos = self.pos + 1               # push right: move one square right
        else:
            self.pos = max(self.pos - 1, 0)       # push left: move one square left, but the wall stops you at S0
        reached_goal = (self.pos == 3)
        reward = 10 if reached_goal else 0
        return self.pos, reward, reached_goal     # SAME AS: the dummy game / CartPole `step` (state, reward, done)


def name(s):
    return "GOAL" if s == 3 else f"S{s}"


def show_table(Q):
    print("    table now:     [push left, push right]")
    for s in (0, 1, 2):
        print(f"      row {name(s)}:      [{Q[(s,0)]:6.3f}, {Q[(s,1)]:6.3f}]")


alpha = 0.5       # WHAT: how far to move toward the answer key (0.5 = halfway). I chose a big number so changes are easy to see.
gamma = 0.9       # WHAT: how much the "still to come" part counts (0.9 = counts 90%)

# WHAT: the Q-table itself: a dictionary, key = (state, action), value = the score. Everything starts at 0.0.
# WHY:  THIS DICTIONARY *IS* THE AGENT'S KNOWLEDGE. Learning = changing the numbers in here.
# BECOMES: in Part 3 it is a numpy array indexed by bucket numbers; in Part 4 it is replaced by a Keras network's weights.
Q = {(s, a): 0.0 for s in (0, 1, 2) for a in (0, 1)}

env_h = Hallway()
for lap in (1, 2, 3):
    print(f"=== LAP {lap} (walker always pushes right) ===")
    state = env_h.reset()
    while True:
        action = 1                                                    # always right, just for this demonstration
        next_state, reward, done = env_h.step(action)

        # ---- JOB 3: THE TARGET (the answer key) ----
        if done:
            best_next = 0.0              # WHAT: reached the goal, game over, nothing can come after it
                                         # SAME AS: CartPole code `if terminated: future = 0.0`
                                         # BECOMES: DQN uses `* (1.0 - D)` for the same job. A2C uses `0.0 if last_terminated`
        else:
            # WHAT: look in the NEXT state's row and take the larger of its two numbers
            # SAME AS: CartPole code `np.max(Q[next_state])`
            best_next = max(Q[(next_state, 0)], Q[(next_state, 1)])
        target = reward + gamma * best_next      # WHAT: the answer key = real reward just seen + discounted best-in-next-row
                                                 # WHY: "better than the old number" = this contains ONE REAL FACT (the reward)
                                                 #      that the old number didn't have. That's the entire source of learning.
                                                 # SAME AS: DQN line `R + gamma * q_next.max(axis=1) * (1.0 - D)`

        # ---- JOB 4: THE UPDATE ----
        old = Q[(state, action)]                 # WHAT: read the ONE cell we are about to change
        new = old + alpha * (target - old)       # WHAT: move that cell a fraction `alpha` of the way toward the answer key
                                                 # WHY:  same as  new = (1 - alpha)*old + alpha*target, a BLEND of the old
                                                 #       belief and the new evidence. The old value is NOT thrown away.
                                                 # BECOMES: DQN: `model.train_on_batch(S, targets)` nudges every weight so the
                                                 #          output moves toward the target. alpha turns into the optimizer's learning_rate.
        Q[(state, action)] = new                 # WHAT: write it back. ONLY this one cell changes; all other cells stay as they were.

        print(f"  {name(state)} --push right--> {name(next_state)} | reward {reward}"
              f" | target = {reward} + {gamma} * {best_next:.3f} = {target:.3f}"
              f" | cell [{name(state)}, right]: {old:.3f} -> {new:.3f}")
        state = next_state
        if done:
            break
    show_table(Q)

**How to read that output.**
- **Lap 1:** the first two steps change nothing, because every number in the table is 0 and the reward is 0, so the target is 0. At the last step the walker gets the real **+10**, and cell `[S2, right]` becomes 5. *This is the only place real information enters the table.*
- **Lap 2:** now `S1 → S2` has a target of `0 + 0.9 × 5 = 4.5`, so `[S1, right]` moves. The +10 is leaking backward, one square per lap.
- **Lap 3:** `[S0, right]` finally becomes nonzero.

That backward leak is how a number that only gets an outside reward at the very end ends up informing decisions at the very beginning. Now let the walker learn on its own with some random exploring (job 5), and see whether the table ends up as a good policy, using the **Part 1 ruler** idea (here the score is "how many steps to reach the goal", where fewer is better).

In [ ]:
def best_action(Q, state):
    # WHAT: the POLICY made from the table: look at this state's row and pick the column with the bigger number.
    # WHY:  this function is what we plug into the "action = policy(state)" slot from Part 1.
    # SAME AS: CartPole's `best_action_np`, and DQN's `np.argmax(model(state))` (a network's two outputs play the role of the row)
    if Q[(state, 0)] > Q[(state, 1)]:
        return 0
    if Q[(state, 1)] > Q[(state, 0)]:
        return 1
    return random.choice([0, 1])             # WHAT: a tie (e.g. at the start, all zeros) is broken by a coin flip


def train_hallway(episodes, epsilon, alpha, gamma):
    Q = {(s, a): 0.0 for s in (0, 1, 2) for a in (0, 1)}
    env = Hallway()
    for _ in range(episodes):
        state = env.reset()
        while True:
            # ---- JOB 5: EXPLORATION ----
            if random.random() < epsilon:
                action = random.choice([0, 1])   # WHAT: with probability epsilon, ignore the table and try something random
                                                 # WHY:  without this the walker could get stuck repeating an early habit and
                                                 #       never discover better options
                                                 # BECOMES: DQN uses the same epsilon idea. A2C has no epsilon: its policy is
                                                 #          already random by design (it samples from probabilities)
            else:
                action = best_action(Q, state)   # WHAT: otherwise trust the table
            next_state, reward, done = env.step(action)
            best_next = 0.0 if done else max(Q[(next_state, 0)], Q[(next_state, 1)])   # SAME AS the target lines above
            target = reward + gamma * best_next
            Q[(state, action)] = Q[(state, action)] + alpha * (target - Q[(state, action)])   # SAME AS the update lines above
            state = next_state
            if done:
                break
    return Q


def steps_to_goal(policy, max_steps=200):
    env = Hallway()
    state = env.reset()
    for step in range(1, max_steps + 1):
        state, reward, done = env.step(policy(state))
        if done:
            return step
    return max_steps


random.seed(0)
Q_hall = train_hallway(episodes=500, epsilon=0.3, alpha=0.5, gamma=0.9)
show_table(Q_hall)

coin_steps = np.mean([steps_to_goal(lambda s: random.choice([0, 1])) for _ in range(1000)])
learned_steps = np.mean([steps_to_goal(lambda s: best_action(Q_hall, s)) for _ in range(1000)])
print()
print(f"coin-flip walker (the BEFORE)  needs on average {coin_steps:5.1f} steps to reach the goal")
print(f"learned-table walker (AFTER)   needs on average {learned_steps:5.1f} steps to reach the goal (3 is the minimum possible)")

**What to notice.**
- In the final table, the **right column** should be close to **8.1, 9, 10** for S0, S1, S2 (each is `0.9 ×` the next), and bigger than the left column. That table *is* the policy: "always push right." The walker discovered it from nothing but a +10 at the end.
- Compare the two step counts. The coin-flip walker wanders. The learned walker takes the shortest route. That is exactly the Part 1 idea (before → after), in a game small enough to verify by hand.

### Why move only *part* of the way? The walker can't move half a square

The *walker* always moves one whole square. What moves partially is **the number in the table cell**, the walker's *belief* about how good a move is. In the hallway above the outcomes are perfectly predictable, so moving all the way (`alpha = 1`) would work too. Partial steps matter when **one observation can mislead you**. Picture a cell where the move pays 10 seventy percent of the time and 0 otherwise, so the true average is 7. Watch what the two settings do:

In [ ]:
random.seed(3)

cell_full = 0.0     # a cell updated with alpha = 1.0 ("jump all the way to what I just saw")
cell_part = 0.0     # a cell updated with alpha = 0.1 ("blend: 90% old belief, 10% new evidence")
full_history = []
part_history = []

for i in range(300):
    reward = 10 if random.random() < 0.7 else 0     # WHAT: a noisy outcome. Pays 10 with 70% chance, else 0. True average: 7.
    cell_full = cell_full + 1.0 * (reward - cell_full)   # WHAT: alpha = 1.0 -> new value is simply the latest reward
    cell_part = cell_part + 0.1 * (reward - cell_part)   # WHAT: alpha = 0.1 -> small blend toward the latest reward
    full_history.append(cell_full)
    part_history.append(cell_part)

print("last 10 values of the cell with alpha = 1.0:", [round(x, 1) for x in full_history[-10:]])
print("last 10 values of the cell with alpha = 0.1:", [round(x, 1) for x in part_history[-10:]])
print()
print(f"over the last 100 updates, alpha = 1.0 swung between {min(full_history[-100:]):.1f} and {max(full_history[-100:]):.1f}"
      f"   (average {np.mean(full_history[-100:]):.1f})")
print(f"over the last 100 updates, alpha = 0.1 stayed between {min(part_history[-100:]):.1f} and {max(part_history[-100:]):.1f}"
      f"   (average {np.mean(part_history[-100:]):.1f})")
print("true average of this noisy outcome: 7.0")
# WHY: alpha = 1.0 just copies the most recent outcome, so it flips between 0 and 10 and is never a usable estimate.
#      alpha = 0.1 averages over many recent outcomes, so it hovers in a much narrower band around the true 7 (not exactly 7:
#      it still drifts a bit, which is the price of a small alpha reacting slowly but steadily).
#      In CartPole one table row stands for MANY slightly different real situations (buckets blur them), so the same
#      row + action gives different results, and the partial update is what averages them.

So `alpha` is a **trust dial**: how much one new observation can overwrite what you already believed. High alpha trusts the latest outcome; low alpha trusts the accumulated history. (`gamma` and `epsilon` are the other two dials. They are called **hyperparameters**: settings *you* choose, never learned.)

# Part 3: Real CartPole with a Q-table

> **You are here:** you've seen the update rule work on a 3-state hallway.
> **New idea:** CartPole's state is four *decimal* numbers, so there are infinitely many possible states and a table can't have a row for each. Fix: **buckets** (chop each number into a few ranges, and say which range it's in).
> **Same as before:** the five jobs and the update rule, line for line.

CartPole: a pole balanced on a cart. Each tick you push the cart left (0) or right (1). The state is four numbers: **cart position, cart velocity, pole angle, pole angular velocity**. You get +1 per tick the pole stays up. The game ends if the pole tilts more than about 12° or the cart leaves the track, or after 500 ticks (so **500 is a perfect score**).

First the "before" score (the coin flip) and a hand-written reference rule, measured with a ruler just like Part 1's `play_one_game`:

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")      # WHAT: create the game
obs, info = env.reset(seed=0)      # WHAT: start a game. `obs` = the 4 numbers: [cart position, cart velocity, pole angle, pole angular velocity]
                                   # SAME AS: Part 1 `game.reset()` (hands back the first state, but now 4 numbers)


def evaluate_cartpole(policy, num_games=100):
    # WHAT: the CartPole version of Part 1's `play_one_game`: play games with a policy, return the average score.
    # WHY:  the same ruler for EVERY method below (coin flip, table, DQN, A2C), so scores are comparable.
    scores = []
    for _ in range(num_games):
        obs, info = env.reset()
        total = 0
        while True:
            action = policy(obs)                       # SAME AS: Part 1 `action = policy(state)`. THE PLUG-IN POINT again.
            obs, reward, terminated, truncated, info = env.step(action)
            total = total + reward
            if terminated or truncated:
                break
        scores.append(total)
    return float(np.mean(scores))


def cartpole_coin_flip(obs):
    return random.choice([0, 1])             # SAME AS: Part 1 `coin_flip_policy`. The "BEFORE" score.


def cartpole_hand_rule(obs):
    angle, angle_speed = obs[2], obs[3]
    return 1 if (angle + angle_speed) > 0 else 0   # WHAT: push toward the side the pole is leaning/tipping toward
                                                   # SAME AS: Part 1 `hand_rule_policy`. It uses 2 of the 4 numbers,
                                                   #          and shows a good policy exists. Nobody hands us this in real problems.

random.seed(0)
before = evaluate_cartpole(cartpole_coin_flip)
rule = evaluate_cartpole(cartpole_hand_rule)
print(f"coin flip (BEFORE) : {before:6.1f} out of 500")
print(f"hand rule          : {rule:6.1f} out of 500   <- proof that a high score is possible")

**Bucketing.** `np.digitize(x, edges)` says which interval `x` falls into, given cut points (`edges`). I use more buckets for the pole angle and pole speed (they matter most) and fewer for the cart's position and speed. A **state** in our table becomes a *tuple of four bucket numbers* such as `(2, 2, 4, 3)`, which is a legitimate row label.

In [ ]:
edges = [
    np.array([-1.2, -0.4, 0.4, 1.2]),            # WHAT: cut points for cart position       -> 5 buckets
    np.array([-1.0, -0.3, 0.3, 1.0]),            # WHAT: cut points for cart velocity       -> 5 buckets
    np.linspace(-0.21, 0.21, 9)[1:-1],           # WHAT: cut points for pole angle          -> 8 buckets
    np.linspace(-2.0, 2.0, 9)[1:-1],             # WHAT: cut points for pole angular speed  -> 8 buckets
]
bucket_counts = [len(e) + 1 for e in edges]      # WHAT: [5, 5, 8, 8]. Rows in the table = 5*5*8*8 = 1600

def to_bucket(obs):
    # WHAT: turn 4 decimals into 4 bucket numbers, e.g. [0.01, -0.02, 0.005, 0.04] -> (2, 2, 4, 4)
    # WHY:  THIS FUNCTION IS THE ONLY NEW IDEA IN THIS PART. After it, the state is a tuple and the Part 2 table logic works.
    # BECOMES: in Part 4 (Keras) this function disappears: the network reads the 4 raw decimals directly.
    return tuple(int(np.digitize(obs[i], edges[i])) for i in range(4))

print("rows in the table:", int(np.prod(bucket_counts)))
obs, info = env.reset()
print("a real observation :", np.round(obs, 3))
print("its row (buckets)  :", to_bucket(obs))



Now the full training loop. It is the Part 2 hallway code with **three changes**: the state is `to_bucket(obs)`; the table is a numpy array of shape `(5, 5, 8, 8, 2)`, where indexing with `state + (action,)` picks one cell; and `epsilon` shrinks over time. Compare line by line with the hallway version: the comments quote it.

Here is that cell piece by piece, with real numbers.

**What the cell does.** The Q-table needs a finite number of rows, but each CartPole number is a decimal with endless possible values. So each number gets sorted into a bucket, like putting ages into "0-9, 10-19, 20-29...". The code below defines where the bucket boundaries are.

**`edges` is four lists of cut points, one list per state number.** Take the cart position: `[-1.2, -0.4, 0.4, 1.2]`. Those four cuts split the number line into five buckets:

```
 below -1.2 | -1.2 to -0.4 | -0.4 to 0.4 | 0.4 to 1.2 | above 1.2
   bucket 0       bucket 1      bucket 2     bucket 3     bucket 4
```

Bucket 2 means "near the center of the track". So n cuts make n+1 buckets, which is what `len(e) + 1` computes. That line gives `[5, 5, 8, 8]`.

**The `linspace` lines just generate evenly spaced cuts so I didn't have to type them.** `np.linspace(-0.21, 0.21, 9)` makes 9 evenly spaced numbers from -0.21 to 0.21: `-0.21, -0.1575, -0.105, -0.0525, 0, 0.0525, 0.105, 0.1575, 0.21`. The `[1:-1]` drops the first and last, leaving 7 cuts. That makes 8 buckets for the pole angle. I drop the ends because -0.21 and 0.21 are about where the pole counts as fallen, so the outermost buckets simply mean "anything beyond the last cut".

**`np.digitize(x, cuts)` tells you which bucket `x` falls in.** It returns how many cut points `x` is past. Using your example `[0.01, -0.02, 0.005, 0.04]`:

| Number | Value | Cuts it is above | Bucket |
|---|---|---|---|
| cart position | 0.01 | -1.2 and -0.4 (2 cuts) | 2 |
| cart velocity | -0.02 | -1.0 and -0.3 (2 cuts) | 2 |
| pole angle | 0.005 | -0.1575, -0.105, -0.0525, 0 (4 cuts) | 4 |
| pole angular speed | 0.04 | -1.5, -1, -0.5, 0 (4 cuts) | 4 |

So that observation becomes the state `(2, 2, 4, 4)`.

**The one line in `to_bucket`, written out the long way:**

```python
buckets = []
for i in range(4):                               # go through the 4 numbers one at a time
    number = obs[i]                              # the i-th state number
    cuts = edges[i]                              # the cut list that belongs to that number
    bucket = int(np.digitize(number, cuts))      # which bucket is it in? (int() makes it a plain integer)
    buckets.append(bucket)
state = tuple(buckets)                           # freeze the list into a tuple, e.g. (2, 2, 4, 4)
```

The `tuple(... for i in range(4))` version does the same thing in one line.

**Why a tuple?** The tuple works as the row label for the Q-table. In the training loop, `state + (action,)` just glues the action on the end, so `(2, 2, 4, 4) + (1,)` becomes `(2, 2, 4, 4, 1)`. That picks one cell in a table of shape `(5, 5, 8, 8, 2)`, which is one cell per state per action.

**Why 1600 rows?** 5 × 5 × 8 × 8 = 1600, one row for every possible combination of the four buckets.

**The last three lines** reset the game, print the real observation, and print which row it landed in. That's just a demonstration that the function works.

The cut points are my choice, and so is how many buckets each number gets. I used more buckets for pole angle and speed because they matter most. If you ask why those numbers, the honest answer is that they're tuning knobs, like alpha and gamma.

In [ ]:
def best_action_np(Q, state):
    row = Q[state]                                   # WHAT: the 2 numbers for this row: [score for left, score for right]
    if row[0] == row[1]:
        return random.choice([0, 1])                 # SAME AS: hallway `best_action` tie -> coin flip
    return int(np.argmax(row))                       # SAME AS: hallway `best_action` pick the bigger number


def train_q_cartpole(episodes, alpha=0.1, gamma=0.99):
    # WHAT: alpha = 0.1 (small steps, because one row blurs many real situations; see the noisy-cell demo in Part 2)
    # WHAT: gamma = 0.99 (CartPole needs foresight over hundreds of ticks, so the future should count for almost as much as now)
    Q = np.zeros(tuple(bucket_counts) + (2,))        # SAME AS: hallway `Q = {(s,a): 0.0 ...}`. The knowledge starts at all zeros.
    scores = []
    for ep in range(episodes):
        epsilon = max(0.01, 1.0 - ep / (episodes * 0.6))   # WHAT: starts at 1.0 (= pure coin flip, the "BEFORE") and shrinks to 0.01
                                                           # WHY: early on the table knows nothing, so explore; later trust it
        obs, info = env.reset()
        state = to_bucket(obs)                       # NEW: the only change from the hallway: state = bucket tuple
        total = 0
        while True:
            if random.random() < epsilon:            # SAME AS: hallway `if random.random() < epsilon`
                action = random.choice([0, 1])
            else:
                action = best_action_np(Q, state)

            obs, reward, terminated, truncated, info = env.step(action)   # JOB 1 (environment loop)
            next_state = to_bucket(obs)

            if terminated:
                future = 0.0                         # SAME AS: hallway `if done: best_next = 0.0` (pole fell, nothing comes after)
            else:
                future = np.max(Q[next_state])       # SAME AS: hallway `max(Q[(next_state,0)], Q[(next_state,1)])`
            target = reward + gamma * future         # SAME AS: hallway `target = reward + gamma * best_next`  (here reward is always 1)

            old = Q[state + (action,)]               # SAME AS: hallway `old = Q[(state, action)]`
            Q[state + (action,)] = old + alpha * (target - old)   # SAME AS: hallway `new = old + alpha*(target-old)`. ONE cell changes.

            state = next_state
            total = total + reward
            if terminated or truncated:
                break

        scores.append(total)
        if (ep + 1) % 1000 == 0:
            print(f"games {ep-998:5d}-{ep+1:5d}: average training score {np.mean(scores[-1000:]):6.1f}")
    return Q, scores


random.seed(0)
Q_cart, table_scores = train_q_cartpole(3000)

def table_policy(obs):
    return best_action_np(Q_cart, to_bucket(obs))    # WHAT: the learned policy: bucket the state, pick the bigger number in that row
                                                     # SAME AS: Part 1's plug-in slot, now filled with a LEARNED rule

table_test = evaluate_cartpole(table_policy)
print(f"\ntable, tested on 100 fresh games (no random exploring): {table_test:.1f} out of 500")
print(f"BEFORE (coin flip): {before:.1f}   |   AFTER learning: {table_test:.1f}   |   hand rule (reference): {rule:.1f}")

**What to notice.** The training lines climb from near the coin-flip level toward hundreds. The test score at the end is typically well below 500 and below my hand rule, because the buckets throw away detail (a pole angle of 0.001 and 0.020 can land in the same bucket) and the table had to *discover* which numbers matter. The important result is the **jump from the before number to the after number**, with no one writing the rule. In a Keras project this table is replaced by a neural network, which is the next part.

# Part 4: DQN in Keras ("Q-learning with a neural network")

> **You are here:** a table-based Q-learner that works but needed buckets.
> **New idea:** replace the *table* with a *neural network* that takes the 4 raw numbers and outputs 2 scores (Q for left, Q for right). No buckets needed.
> **Same as before:** jobs 1 to 5, the target formula, epsilon exploration. **Only job 4 (the update) really changes.**

**What a Keras network is, in this context.** `Dense(24)` is a layer of 24 weighted sums of its inputs (each with adjustable weights). Stack a few of them and the last layer outputs 2 numbers. The network is a **function** with lots of adjustable numbers (weights) inside, so "learning" means adjusting the weights. It replaces `Q[state]`: you hand it a state, it hands back `[Q(left), Q(right)]`, *and it works for states it never saw, because similar inputs give similar outputs.*

**Two extras that real DQNs add**:
- **Replay memory:** store each step `(state, action, reward, next_state, terminated)` in a list, and train on **random batches** from it. Consecutive steps are nearly identical, so training on them in order is unstable; random batches fix that, and old steps get reused.
- **Target network:** a slowly updated *copy* of the network that is used only to compute targets, so the answer key doesn't change on every update.

In [ ]:
import keras

keras.utils.set_random_seed(0)

def build_q_network():
    # WHAT: a network with 4 inputs (the raw CartPole numbers) and 2 outputs (score for left, score for right)
    net = keras.Sequential([
        keras.layers.Input(shape=(4,)),                 # WHAT: the 4 state numbers go in. NO bucketing needed.
        keras.layers.Dense(24, activation="relu"),      # WHAT: 24 weighted sums, then "relu" (keep positives, zero out negatives)
        keras.layers.Dense(24, activation="relu"),      # WHY: stacking layers lets the function bend (represent curves, not just lines)
        keras.layers.Dense(2, activation="linear"),     # WHAT: the 2 outputs = [Q(left), Q(right)]  SAME AS: one row of the table
    ])
    # WHAT: compile = tell Keras HOW to learn: measure error with mean squared error, adjust weights with the Adam optimizer
    # WHY:  "mse" = the average of (target - output)^2. Making it small = moving outputs toward targets.
    # SAME AS: table update `new = old + alpha*(target - old)`, which is also "move toward target". Here learning_rate plays alpha's role.
    net.compile(loss="mse", optimizer=keras.optimizers.Adam(learning_rate=0.001))
    return net


def train_dqn(episodes):
    train_env = gym.make("CartPole-v1")

    model = build_q_network()                           # SAME AS: `Q = np.zeros(...)`: the agent's knowledge (random weights instead of zeros)
    target_model = build_q_network()                    # WHAT: the slowly-updated copy used only to compute targets
    target_model.set_weights(model.get_weights())

    gamma = 0.99                                        # SAME AS: table version gamma
    epsilon, epsilon_min, epsilon_decay = 1.0, 0.01, 0.99   # SAME AS: table epsilon (1.0 = pure coin flip at the start)
    batch_size = 64
    memory = collections.deque(maxlen=10000)            # WHAT: the replay memory. A list that forgets its oldest items when full.
    sync_every = 500                                    # WHAT: copy the model into the target copy every 500 steps
    steps = 0
    scores = []

    for ep in range(episodes):
        obs, info = train_env.reset()
        total = 0
        while True:
            # ---- JOB 2 + 5: POLICY and EXPLORATION ----
            if random.random() < epsilon:
                action = random.choice([0, 1])                                  # SAME AS: table `random.choice([0, 1])` explore
            else:
                q_values = model(obs[None, :], training=False).numpy()[0]       # WHAT: ask the network for [Q(left), Q(right)]
                                                                                # SAME AS: table `Q[state]` (reading a row)
                action = int(np.argmax(q_values))                               # SAME AS: table `np.argmax(row)` (pick the bigger one)

            # ---- JOB 1: ENVIRONMENT LOOP ----
            next_obs, reward, terminated, truncated, info = train_env.step(action)

            # WHAT: remember this step. WATCH: store `terminated` (the pole fell), NOT `truncated` (time ran out).
            #       Running out of time isn't a failure, so the future still counts.
            memory.append((obs, action, reward, next_obs, float(terminated)))
            obs = next_obs
            total = total + reward
            steps = steps + 1

            # ---- JOBS 3 + 4: TARGET and UPDATE (only once the memory has enough steps) ----
            if len(memory) >= batch_size:
                batch = random.sample(memory, batch_size)                       # WHAT: 64 RANDOM old steps (not the latest ones)
                S  = np.array([b[0] for b in batch], dtype=np.float32)          # states
                A  = np.array([b[1] for b in batch])                            # actions taken
                R  = np.array([b[2] for b in batch], dtype=np.float32)          # rewards
                S2 = np.array([b[3] for b in batch], dtype=np.float32)          # next states
                D  = np.array([b[4] for b in batch], dtype=np.float32)          # 1.0 if the pole fell on that step, else 0.0

                q_now = model(S, training=False).numpy()                        # WHAT: current guesses for all 64 states: shape (64, 2)
                q_next = target_model(S2, training=False).numpy()               # WHAT: the copy's guesses for the NEXT states: shape (64, 2)

                targets = q_now.copy()                                          # WHAT: start the answer key = current guesses...
                # WHAT: ...then replace ONLY the output for the action that was actually taken with the new target.
                # SAME AS: table `target = reward + gamma * np.max(Q[next_state])`
                #          with `if terminated: future = 0.0` handled by the `* (1.0 - D)` (D = 1 means "fell", so the future term vanishes).
                # WHY:     we only have evidence about the action we took; the other output keeps its current value so it gets NO push.
                targets[np.arange(batch_size), A] = R + gamma * q_next.max(axis=1) * (1.0 - D)

                # WHAT: nudge ALL the network's weights so outputs move toward `targets` (this is the "fit")
                # SAME AS: table `Q[state + (action,)] = old + alpha * (target - old)`
                # BECOMES: (nothing new; this is the big change from the table: ONE cell moves vs MANY weights move.)
                # WHY:     because weights are shared across states, learning from one state also changes the guesses for
                #          SIMILAR states. That generalization is what the table could not do.
                model.train_on_batch(S, targets)

            if steps % sync_every == 0:
                target_model.set_weights(model.get_weights())                   # WHAT: refresh the answer-key copy now and then

            if terminated or truncated:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)                     # SAME AS: table's shrinking epsilon
        scores.append(total)
        if (ep + 1) % 50 == 0:
            print(f"episodes {ep-48:4d}-{ep+1:4d}: average training score {np.mean(scores[-50:]):6.1f}   (epsilon now {epsilon:.2f})")
    return model, scores


random.seed(0)
dqn_model, dqn_scores = train_dqn(300)

In [ ]:
def dqn_policy(obs):
    # WHAT: the learned DQN policy: ask the network for [Q(left), Q(right)] and take the bigger. No epsilon when just playing.
    # SAME AS: Part 3 `table_policy`, with the network's two outputs standing in for the table's row.
    return int(np.argmax(dqn_model(obs[None, :], training=False).numpy()[0]))

random.seed(1)
dqn_test = evaluate_cartpole(dqn_policy, num_games=20)
print(f"DQN tested on 20 fresh games (no exploring): {dqn_test:.1f} out of 500")
print(f"BEFORE (coin flip): {before:.1f}   |   DQN AFTER learning: {dqn_test:.1f}   |   hand rule (reference): {rule:.1f}")

**What to notice.** The training lines should climb well past the coin-flip level, but they can **wobble** (a good stretch followed by a worse one). That is typical for DQN, not a bug in the code. And notice what changed relative to the table: no `to_bucket`, and the update is "nudge every weight" instead of "change one cell". The *target* line is the same idea word for word.

Now the other method to compare it with.

# Part 5: A2C in Keras

> **You are here:** DQN learns *how good each action is*, then picks the best (the policy is derived from the numbers).
> **New idea:** A2C learns the **policy itself**, with two networks:
> - the **actor** takes a state and outputs a *probability* for each action (e.g. 70% left, 30% right). The agent *rolls dice* with those probabilities. The actor **is** the policy.
> - the **critic** takes a state and outputs *one number*: "about how many points will I collect from here?" It never picks actions. It judges situations.
> **Same as before:** jobs 1 to 5, and the idea of a *target*. **What's new:** the **advantage**, and the fact that the update is no longer a plain "move toward target".

**Advantage = target − critic's guess.** The critic guessed some value for the state you were in. The target (the real rewards you then collected, plus the critic's guess about the rest) says what it was *actually* worth. If the actual is higher, **advantage > 0**: "that went better than expected, so make the action I took more likely." If lower, **advantage < 0**: "worse than expected, make it less likely."

**What "A2C" adds to this idea:** the two tables become two networks, and instead of updating after every single step it collects a short run of steps (here 8) and updates on that batch. The real A2C also usually runs several games in parallel; I use one to keep it readable.

**One honest note.** My first attempt at this collapsed (scores fell to about 10, because the policy locked onto always pushing one way). Two things I changed together made it learn: normalizing the advantages and making the critic learn faster, plus using 8-step returns. I did not test which one mattered most. This is typical: actor-critic methods are sensitive to their settings.

In [ ]:
import tensorflow as tf     # WHY: the actor's update isn't a plain "mse toward a target", so we write it ourselves with TensorFlow's tools

keras.utils.set_random_seed(0)
np.random.seed(0)

actor = keras.Sequential([
    keras.layers.Input(shape=(4,)),                      # WHAT: the 4 raw state numbers in (no buckets)
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(2, activation="softmax"),         # WHAT: 2 outputs that are PROBABILITIES adding to 1, e.g. [0.7, 0.3]
                                                         # WHY: "softmax" turns any 2 numbers into probabilities. THIS output is the policy.
])
critic = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1),                               # WHAT: ONE number = the critic's guess of "points still to come from this state"
                                                         # SAME AS: the table version's V (and like DQN's output, but for the STATE, not per action)
])
actor_optimizer = keras.optimizers.Adam(learning_rate=0.0005)    # WHAT: how the actor's weights get nudged (slow, steady)
critic_optimizer = keras.optimizers.Adam(learning_rate=0.003)    # WHAT: critic learns faster so its guesses are usable early
gamma = 0.99                                             # SAME AS: DQN / table gamma
entropy_coef = 0.01                                      # WHAT: weight of a small bonus for keeping probabilities spread out (explained below)
n_steps = 8                                              # WHAT: how many steps to collect before each update


@tf.function
def a2c_update(S, A, targets):
    # WHAT: S = a batch of states, A = the actions taken there, targets = the "answer key" numbers for each state.
    # WATCH: `tf.GradientTape` records the arithmetic so TensorFlow can work out which way to nudge each weight.
    #        In DQN, `model.train_on_batch` did all of this for us because its loss was plain "mse".
    #        Here the actor's loss is custom, so we write the update ourselves. That is the main code difference between DQN and A2C.

    # ---- CRITIC UPDATE (same job as DQN's update) ----
    with tf.GradientTape() as critic_tape:
        values = tf.squeeze(critic(S), axis=1)                    # WHAT: critic's current guesses for these states
        critic_loss = tf.reduce_mean(tf.square(targets - values)) # WHAT: average squared gap between answer key and guess
                                                                  # SAME AS: DQN `loss="mse"`, and table `(target - old)`: "how far off was I?"
    critic_grads = critic_tape.gradient(critic_loss, critic.trainable_variables)
    critic_optimizer.apply_gradients(zip(critic_grads, critic.trainable_variables))   # WHAT: nudge critic weights to shrink that gap
                                                                                      # SAME AS: DQN `model.train_on_batch(S, targets)`

    # ---- ADVANTAGE ----
    advantage = tf.stop_gradient(targets - values)
    # WHAT: advantage = answer key - critic's guess  ("was it better or worse than expected?")
    # WHY:  `stop_gradient` means "treat this as a fixed number; don't nudge the critic via the actor's loss".
    #       It is a SCORE that tells the actor which way to move, not something to adjust.
    # SAME AS: table-version `adv = target - V[state]`
    advantage = (advantage - tf.reduce_mean(advantage)) / (tf.math.reduce_std(advantage) + 1e-8)
    # WHAT: rescale advantages in this batch to average 0 and spread 1.
    # WHY:  keeps update sizes consistent so learning isn't thrown off by occasional huge advantages (one of the two things that fixed my first attempt).

    # ---- ACTOR UPDATE (the NEW part; DQN has nothing like this) ----
    with tf.GradientTape() as actor_tape:
        probs = actor(S)                                                       # WHAT: the actor's probabilities, shape (batch, 2)
        prob_of_taken = tf.reduce_sum(probs * tf.one_hot(A, 2), axis=1)        # WHAT: pick out the probability of the action we ACTUALLY took
        log_prob = tf.math.log(prob_of_taken + 1e-8)                           # WHAT: the log of it (a standard trick that makes the math behave)
        entropy = -tf.reduce_sum(probs * tf.math.log(probs + 1e-8), axis=1)    # WHAT: how "spread out" the probabilities are (bigger = more random)
        actor_loss = -tf.reduce_mean(log_prob * advantage) - entropy_coef * tf.reduce_mean(entropy)
        # WHAT: loss = -(log probability of the action taken) * (advantage), then a small entropy bonus.
        # WHY:  Keras always tries to make the loss SMALLER.
        #       If advantage > 0, making log_prob bigger makes the loss smaller -> the actor makes that action MORE likely.
        #       If advantage < 0, the opposite -> LESS likely.
        # SAME AS: table-version `H[state] += alpha_actor * advantage * grad` ("raise preference for good surprises, lower for bad").
        # WHY the entropy term: a small reward for staying random keeps the policy exploring.
        # SAME AS (exploration): DQN's epsilon, but built into the policy instead of added on top.
    actor_grads = actor_tape.gradient(actor_loss, actor.trainable_variables)
    actor_optimizer.apply_gradients(zip(actor_grads, actor.trainable_variables))


def train_a2c(num_episodes):
    train_env = gym.make("CartPole-v1")
    obs, info = train_env.reset()
    episode_total = 0
    scores = []
    next_report = 50                                          # WHAT: print a progress line every 50 finished games

    while len(scores) < num_episodes:
        S, A, R = [], [], []                                  # WHAT: the short run of steps we collect before each update
        last_terminated = False
        last_next_obs = obs

        # ---- JOBS 1 + 2 + 5: collect up to n_steps steps with the actor's current policy ----
        for _ in range(n_steps):
            probs = actor(obs[None, :], training=False).numpy()[0]   # WHAT: the actor's probabilities for this state, e.g. [0.7, 0.3]
            probs = probs / probs.sum()                              # WATCH: tiny rounding fix so they add to exactly 1
            action = int(np.random.choice(2, p=probs))               # WHAT: ROLL DICE using those probabilities
                                                                     # WHY:  this is the policy AND the exploration, in one line.
                                                                     # DIFFERENT FROM DQN: no `argmax` (always pick the best) and no epsilon.
            next_obs, reward, terminated, truncated, info = train_env.step(action)
            S.append(obs); A.append(action); R.append(reward)
            episode_total = episode_total + reward
            last_terminated = terminated
            last_next_obs = next_obs
            obs = next_obs
            if terminated or truncated:
                scores.append(episode_total)
                episode_total = 0
                obs, info = train_env.reset()
                break
        # IMPORTANT: unlike DQN, these steps are used ONCE and thrown away (no replay memory).
        #            WHY: the actor changes every update, so old steps came from an OLDER policy and are stale.
        #            This is called ON-POLICY learning. DQN with a replay memory is OFF-POLICY.

        # ---- JOB 3: THE TARGET (answer key), computed backwards through the collected steps ----
        if last_terminated:
            running = 0.0                                            # SAME AS: `if terminated: future = 0.0` (the pole fell, nothing comes after)
        else:
            running = float(critic(last_next_obs[None, :], training=False).numpy()[0, 0])
            # WHAT: the run stopped but the game didn't, so use the CRITIC's guess for what comes after the last step
            # WHY:  this is where the critic's number gets mixed into the answer key
            # SAME AS: table-version `future = V[next_state]`
        targets = np.zeros(len(R), dtype=np.float32)
        for t in reversed(range(len(R))):
            running = R[t] + gamma * running                         # WHAT: this step's reward + discounted everything after it
            targets[t] = running
        # SAME AS: DQN `R + gamma * q_next.max(...)`, but looking up to 8 REAL rewards ahead before trusting the critic.
        # WHY: with n_steps = 1 this is exactly "reward + gamma * V(next_state)". Using 8 means real rewards count more
        #      and the critic's still-bad early guesses count less.

        # ---- JOB 4: THE UPDATE (critic + actor, as written in a2c_update above) ----
        a2c_update(np.array(S, dtype=np.float32), np.array(A), targets)

        if len(scores) >= next_report:
            print(f"episodes {next_report-49:4d}-{next_report:4d}: average training score {np.mean(scores[-50:]):6.1f}")
            next_report = next_report + 50
    return scores


a2c_scores = train_a2c(500)

In [ ]:
def a2c_policy(obs):
    # WHAT: the A2C policy: get the actor's probabilities and roll dice with them.
    # WHY:  for A2C the policy IS the probabilities, so we test it the way it plays. (DQN's version used argmax.)
    probs = actor(obs[None, :], training=False).numpy()[0]
    return int(np.random.choice(2, p=probs / probs.sum()))

random.seed(1)
np.random.seed(1)
a2c_test = evaluate_cartpole(a2c_policy, num_games=20)
print(f"A2C tested on 20 fresh games: {a2c_test:.1f} out of 500")
print(f"BEFORE (coin flip): {before:.1f}   |   A2C AFTER learning: {a2c_test:.1f}   |   hand rule (reference): {rule:.1f}")

# Part 6: Putting it together

First the numbers from this run, then the side-by-side comparison.

In [ ]:
import matplotlib.pyplot as plt

def moving_average(x, w=25):
    # WHAT: average of every 25 consecutive games, so the line is readable instead of jagged
    x = np.array(x, dtype=float)
    return np.convolve(x, np.ones(w) / w, mode="valid")

plt.figure(figsize=(9, 4.5))
plt.plot(moving_average(table_scores), label="Q-table (buckets)")
plt.plot(moving_average(dqn_scores), label="DQN (Keras)")
plt.plot(moving_average(a2c_scores), label="A2C (Keras)")
plt.axhline(500, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("games played")
plt.ylabel("score (average of 25 games)")
plt.title("Learning curves on CartPole (one run each)")
plt.legend()
plt.show()

print(f"{'method':22s} {'games trained':>14s} {'avg of last 50 training games':>32s} {'test score':>12s}")
print(f"{'coin flip (BEFORE)':22s} {'0':>14s} {'':>32s} {before:12.1f}")
print(f"{'Q-table':22s} {len(table_scores):14d} {np.mean(table_scores[-50:]):32.1f} {table_test:12.1f}")
print(f"{'DQN (Keras)':22s} {len(dqn_scores):14d} {np.mean(dqn_scores[-50:]):32.1f} {dqn_test:12.1f}")
print(f"{'A2C (Keras)':22s} {len(a2c_scores):14d} {np.mean(a2c_scores[-50:]):32.1f} {a2c_test:12.1f}")
print(f"{'hand rule (reference)':22s} {'':>14s} {'':>32s} {rule:12.1f}")

**How to read this honestly.** Each method was run **once**, with a different number of games, and I tuned the settings by hand (more for A2C than for the others). So this is **not a fair race** and you shouldn't write "A2C beats DQN" or the reverse from it. Also, "test score" is a snapshot of the networks at the very last moment, and the training curves wobble, so a single snapshot can land above or below a method's typical level (look at how far the last few training lines bounce around before trusting one test number). What you *can* say from it: all three improved far past the coin-flip "before" score, learning only from the +1-per-tick reward, and the Keras versions (which read the raw numbers) had no need for buckets. Results for all of them wobble from run to run.

## Side-by-side: the five jobs, three ways

| Job | Q-table | DQN (Keras) | A2C (Keras) |
|---|---|---|---|
| **Where the knowledge lives** | The numbers in the table cells | The weights of one network (outputs `[Q(left), Q(right)]`) | The weights of two networks: an **actor** (outputs action probabilities) and a **critic** (outputs one state value) |
| **2. Policy** (state → action) | Pick the bigger number in the row | Pick the bigger network output | **Roll dice** with the actor's probabilities |
| **5. Exploration** | `epsilon`: sometimes pick randomly | `epsilon`, same idea | Built into the policy (it's already random), plus the entropy bonus |
| **3. Target** (answer key) | `reward + gamma * max Q(next state)` | The same formula, using a slowly updated *copy* of the network | Real rewards over the next 8 steps, then the **critic's** guess for the rest |
| **4. Update** | Move **one cell** a fraction `alpha` toward the target | Nudge **all weights** so output moves toward target (`train_on_batch`, plain mse) | **Critic:** same idea as DQN. **Actor:** raise or lower the probability of the action taken, in proportion to the **advantage** (needs a custom loss) |
| **What it learns from** | Each step, immediately | Random batches from a **replay memory** (old steps are reused: **off-policy**) | The latest 8 steps only, then discarded (**on-policy**) |
| **Handling decimals** | Needs **buckets** | Reads raw numbers | Reads raw numbers |

## One sentence

**Q-learning (and DQN) learns *how good each action is* and gets its policy indirectly by picking the best one; A2C learns the *policy itself* (the actor) and uses a second network (the critic) to judge whether each action turned out better or worse than expected (the advantage).**

## Where to look in any Keras code

| Look for this in your code | It is job... |
|---|---|
| `model.predict(...)` or `model(...)` returning 2 numbers, then `np.argmax` | 2: policy |
| `random.random() < epsilon` | 5: exploration |
| `reward + gamma * np.max(...)` (maybe times `(1 - done)`) | 3: target |
| `model.fit(...)` or `train_on_batch(...)` | 4: update |
| `memory.append(...)` / `random.sample(memory, ...)` | the replay memory |
| a second copy of the model with `set_weights` | the target network |